# Task 3: Correlation Between News Sentiment and Stock Returns

## Overview
In this notebook I apply VADER sentiment analysis to AAPL financial news headlines
and measure the statistical correlation between daily sentiment scores and stock price returns.

VADER (Valence Aware Dictionary and sEntiment Reasoner) was selected because it is 
specifically designed for short social media and financial text — making it ideal for 
analyzing financial news headlines.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from scipy import stats
# Download VADER lexicon
nltk.download('vader_lexicon')

print("All libraries loaded successfully")

In [ ]:
try:
    df_news = pd.read_csv("../data/raw/raw_analyst_ratings.csv")
    df_news['date'] = pd.to_datetime(df_news['date'], format='mixed', utc=True)
    df_news['date_only'] = df_news['date'].dt.date
    print(f"News data loaded successfully: {df_news.shape[0]:,} rows")
    df_news.head()
except FileNotFoundError:
    print("ERROR: raw_analyst_ratings.csv not found in data/raw/")
except Exception as e:
    print(f"ERROR loading news data: {e}")

In [ ]:
try:
    df_stock = pd.read_csv("../data/raw/AAPL.csv")
    df_stock['Date'] = pd.to_datetime(df_stock['Date'])
    df_stock['date_only'] = df_stock['Date'].dt.date
    df_stock['Daily_Return'] = df_stock['Close'].pct_change() * 100
    print(f"Stock data loaded successfully: {df_stock.shape[0]:,} rows")
    df_stock.head()
except FileNotFoundError:
    print("ERROR: AAPL.csv not found in data/raw/")
except Exception as e:
    print(f"ERROR loading stock data: {e}")

## Sentiment Analysis Using VADER
VADER assigns a compound score between -1 (most negative) and +1 (most positive).
Scores above 0.05 are positive, below -0.05 are negative, and between are neutral.

In [ ]:
# Initialize VADER
sia = SentimentIntensityAnalyzer()

# Filter news for AAPL only
df_aapl_news = df_news[df_news['stock'] == 'AAPL'].copy()
print(f"AAPL news articles: {len(df_aapl_news)}")

# Apply sentiment to each headline
df_aapl_news['sentiment_score'] = df_aapl_news['headline'].apply(
    lambda x: sia.polarity_scores(str(x))['compound']
)

print("\nSentiment score statistics:")
print(df_aapl_news['sentiment_score'].describe())
print("\nSample results:")
print(df_aapl_news[['headline', 'sentiment_score']].head(10))

## Date Alignment and Aggregation
Multiple articles may be published on the same day for the same stock.
We calculate the average daily sentiment score to get one value per trading day.
Weekend and holiday articles are excluded since they have no matching trading day.

In [ ]:
# Calculate average daily sentiment score
daily_sentiment = df_aapl_news.groupby('date_only')['sentiment_score'].mean().reset_index()
daily_sentiment.columns = ['date_only', 'avg_sentiment']

print("Daily sentiment calculated:")
print(daily_sentiment.head(10))
print(f"\nTotal days with sentiment: {len(daily_sentiment)}")

In [ ]:
# Merge sentiment with stock returns
df_merged = pd.merge(daily_sentiment, df_stock[['date_only', 'Daily_Return']], 
                     on='date_only', how='inner')

# Remove NaN values
df_merged = df_merged.dropna()

print(f"Merged dataset shape: {df_merged.shape}")
print(df_merged.head(10))

## Pearson Correlation Analysis
We calculate the Pearson correlation coefficient between average daily sentiment scores
and daily stock returns. This measures the linear relationship between the two variables.

In [ ]:
# Calculate Pearson correlation
correlation, p_value = stats.pearsonr(
    df_merged['avg_sentiment'], 
    df_merged['Daily_Return']
)

print("=== CORRELATION ANALYSIS ===")
print(f"Pearson Correlation: {correlation:.4f}")
print(f"P-value: {p_value:.4f}")

if abs(correlation) < 0.2:
    strength = "very weak"
elif abs(correlation) < 0.4:
    strength = "weak"
elif abs(correlation) < 0.6:
    strength = "moderate"
else:
    strength = "strong"

direction = "positive" if correlation > 0 else "negative"
print(f"\nInterpretation: {strength} {direction} correlation")

if p_value < 0.05:
    print("Result is statistically significant (p < 0.05)")
else:
    print("Result is NOT statistically significant (p > 0.05)")

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(df_merged['avg_sentiment'], df_merged['Daily_Return'], 
            alpha=0.5, color='steelblue', edgecolors='white', linewidth=0.5)

# Add trend line
z = np.polyfit(df_merged['avg_sentiment'], df_merged['Daily_Return'], 1)
p = np.poly1d(z)
x_line = np.linspace(df_merged['avg_sentiment'].min(), df_merged['avg_sentiment'].max(), 100)
plt.plot(x_line, p(x_line), "r--", linewidth=2, label='Trend line')

plt.title(f'News Sentiment vs Stock Returns (AAPL)\nPearson r = {correlation:.4f}, p = {p_value:.4f}')
plt.xlabel('Average Daily Sentiment Score (VADER)')
plt.ylabel('Daily Stock Return (%)')
plt.legend()
plt.tight_layout()
plt.show()

## Sentiment Category Analysis
We classify each day as Positive, Neutral, or Negative based on sentiment score
and compare average stock returns across these categories.

In [ ]:
# Classify sentiment
def classify_sentiment(score):
    if score > 0.05:
        return 'Positive'
    elif score < -0.05:
        return 'Negative'
    else:
        return 'Neutral'

df_merged['sentiment_category'] = df_merged['avg_sentiment'].apply(classify_sentiment)

# Average return per category
category_returns = df_merged.groupby('sentiment_category')['Daily_Return'].mean()

plt.figure(figsize=(8, 5))
colors = {'Positive': 'green', 'Neutral': 'grey', 'Negative': 'red'}
bars = plt.bar(category_returns.index, category_returns.values,
               color=[colors[c] for c in category_returns.index])
plt.title('Average Daily Return by News Sentiment Category (AAPL)')
plt.xlabel('Sentiment Category')
plt.ylabel('Average Daily Return (%)')
plt.axhline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

print("\nAverage returns by sentiment category:")
print(category_returns)

## Results and Interpretation
Discussion of correlation strength, direction, and statistical significance.

In [ ]:
print("=== RESULTS INTERPRETATION ===\n")
print(f"Pearson Correlation Coefficient: {correlation:.4f}")
print(f"P-value: {p_value:.4f}\n")

print("STATISTICAL FINDINGS:")
print(f"- Correlation of {correlation:.4f} indicates a {strength} {direction} relationship")
print(f"- {len(df_merged)} trading days analyzed after date alignment\n")

print("TRADING IMPLICATIONS:")
if correlation > 0:
    print("- Positive correlation: days with more positive news tend to see higher returns")
    print("- Strategy: monitor sentiment as a CONFIRMATION signal for long positions")
    print("- When RSI < 40 AND sentiment > 0.05: consider entering a long position")
    print("- When RSI > 70 AND sentiment < -0.05: consider reducing exposure")
else:
    print("- Negative correlation detected: positive news may precede profit-taking")

print("\nRISK MANAGEMENT INSIGHTS:")
print("- Sentiment alone is INSUFFICIENT for trading decisions")
print("- Always combine with technical indicators (RSI, MACD) before acting")
print("- High news volume days increase volatility - reduce position sizes")
print("- Set stop-loss orders at 2-3% below entry regardless of sentiment signal")

print("\nLIMITATIONS:")
print("- Market efficiency means most public sentiment is already priced in")
print("- VADER designed for social media - may misread formal financial language")
print("- Lag effects not fully captured: post-market news affects next day price")
print("- Analysis covers AAPL only - results may differ across sectors")

print("\nSENTIMENT BREAKDOWN:")
for category, avg_ret in avg_returns.items():
    count = len(df_merged[df_merged['sentiment_category'] == category])
    print(f"  {category}: {count} days, avg return = {avg_ret:.3f}%")